# Mixing

![star supported](https://img.shields.io/badge/STAR-supported-blue)
![Vantage supported](https://img.shields.io/badge/Vantage-supported-blue)
![OT supported](https://img.shields.io/badge/OT-supported-blue)
![EVO not tested](https://img.shields.io/badge/EVO-not%20supported-red)

Mixing in PLR is hardware agnostic. You can currently mix before an aspiration or after a dispense by passing the `mix` parameter to the corresponding method call.

A standalone mixing operation will be added in the future.

In [1]:
from collections.abc import Awaitable, Callable
from typing import Optional
from pylabrobot.liquid_handling import LiquidHandler, LiquidHandlerChatterboxBackend
from pylabrobot.liquid_handling.standard import Mix
from pylabrobot.resources import Cor_96_wellplate_360ul_Fb, PLT_CAR_L5AC_A00, STARDeck, TIP_CAR_480_A00, hamilton_96_tiprack_1000uL_filter


async def run_chatterbox_mixing_protocol(
  after_pickup: Optional[Callable[[LiquidHandler], Awaitable[None]]] = None,
) -> None:
  lh = LiquidHandler(backend=LiquidHandlerChatterboxBackend(), deck=STARDeck())
  tip_carrier = TIP_CAR_480_A00(name="tiprack_1")
  tip_carrier[0] = tip_rack = hamilton_96_tiprack_1000uL_filter(name="tiprack_1_0")
  lh.deck.assign_child_resource(tip_carrier, track=1)
  plate_carrier = PLT_CAR_L5AC_A00(name="plate_carrier_1")
  plate_carrier[0] = plate = Cor_96_wellplate_360ul_Fb(name="plate_1")
  lh.deck.assign_child_resource(plate_carrier, track=10)
  mix = Mix(volume=50, repetitions=3, flow_rate=100)

  async with lh:
    await lh.pick_up_tips(tip_rack["A1"])
    if after_pickup is not None:
      await after_pickup(lh)
    await lh.aspirate(plate["A1"], vols=[100], mix=[mix])
    await lh.dispense(plate["A1"], vols=[100], mix=[mix])
    await lh.return_tips()

## Run the scoped mixing example

The function above creates a chatterbox-backed liquid handler and keeps setup, tip pickup, aspiration, dispensing, tip return, and cleanup in one lifespan. It passes one `Mix` object per channel to both `LiquidHandler.aspirate` and `LiquidHandler.dispense`. Run the complete software-only example with:

In [2]:
await run_chatterbox_mixing_protocol()

Setting up the liquid handler.
Picking up tips:
pip#  resource             offset           tip type     max volume (µL)  fitting depth (mm)   tip length (mm)  filter    
  p0: tiprack_1_0_tipspot_A1 0,0,0            HamiltonTip  1065             8                    95.1             Yes       
Aspirating:
pip#  vol(ul)  resource             offset           flow rate  blowout    lld_z       
  p0: 100.0    plate_1_well_A1      0,0,0            None       None       None       
Dispensing:
pip#  vol(ul)  resource             offset           flow rate  blowout    lld_z       
  p0: 100.0    plate_1_well_A1      0,0,0            None       None       None       
Dropping tips:
pip#  resource             offset           tip type     max volume (µL)  fitting depth (mm)   tip length (mm)  filter    
  p0: tiprack_1_0_tipspot_A1 0,0,0            HamiltonTip  1065             8                    95.1             Yes       
Stopping the liquid handler.
